Installing CodeCarbon and Checking Output

In [1]:
!pip install -q codecarbon
!pip install -U bitsandbytes>=0.46.1
!nvidia-smi --query-gpu=name,power.draw --format=csv

name, power.draw [W]
Tesla T4, 10.15 W


4-bit Quantized 7B-Llama

In [2]:
import os
import csv
import glob
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from codecarbon import OfflineEmissionsTracker

model_name = "codellama/CodeLlama-7b-Instruct-hf"
version = "quantized_4bit"
country = "USA"        # must match the other three scripts
num_runs = 5
max_new_tokens = 300
save_dir = "/content/quantized_model"   # saved once to measure storage size

# Same NF4 config the pruning stage uses for QLoRA training
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU found. Set the Colab runtime to T4.")
hardware = torch.cuda.get_device_name(0)

# Load only if this exact model + version isn't already in memory
if globals().get("loaded_config") != (model_name, version):
    if "model" in globals():
        raise RuntimeError("A different model/version is loaded. Restart the session first.")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config,
                                                 device_map="cuda")
    model.eval()
    loaded_config = (model_name, version)
else:
    print("Model already loaded, skipping")

# Model size = storage size of the saved 4-bit weights
if not glob.glob(os.path.join(save_dir, "*.safetensors")):
    model.save_pretrained(save_dir)
size_gb = sum(os.path.getsize(f) for f in glob.glob(os.path.join(save_dir, "*.safetensors"))) / 1e9

print("Version:", version)
print("Hardware:", hardware)
print("Quantization: 4-bit NF4, double quant, FP16 compute")
print(f"Model size on disk: {size_gb:.2f} GB")
print(f"GPU memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

Code = '''from typing import List
def below_zero(operations: List[int]) -> bool:
    """ You're given a list of deposit and withdrawal operations
    on a bank account that starts with zero balance. Your task is to
    detect if at any point the balance of account falls below zero,
    and at that point function should return True.
    """
    balance = 0
    for op in operations:
        balance += op
        if balance < 0:
            return True
    return False'''

prompt = ("Generate Google-style docstring for the following Python function: {code}. "
          "Only output the docstring and function code.").format(code=Code)   # P-2

inputs = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}], add_generation_prompt=True,
    return_tensors="pt", return_dict=True
).to(model.device)
input_tokens = inputs["input_ids"].shape[1]

results_csv = f"results_{version}.csv"
outputs_file = f"docstrings_{version}.txt"

# Warm up once (untimed, untracked)
with torch.no_grad():
    model.generate(**inputs, max_new_tokens=20, do_sample=False, pad_token_id=tokenizer.eos_token_id)

with open(results_csv, "w", newline="") as f, open(outputs_file, "w") as out_f:
    writer = csv.writer(f)
    writer.writerow(["run", "input_tokens", "output_tokens", "latency_s",
                     "emissions_kg", "energy_kwh", "accuracy_PF"])

    for run in range(1, num_runs + 1):
        # Fresh tracker per run, wrapped around generation only
        tracker = OfflineEmissionsTracker(country_iso_code=country, measure_power_secs=1,
                                          save_to_file=False, log_level="error")
        tracker.start()
        torch.cuda.synchronize()
        start = time.perf_counter()
        with torch.no_grad():
            generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                           do_sample=True, temperature=0.7,
                                           eos_token_id=tokenizer.eos_token_id,
                                           pad_token_id=tokenizer.eos_token_id)
        torch.cuda.synchronize()
        latency = time.perf_counter() - start
        emissions_kg = tracker.stop()
        energy_kwh = tracker.final_emissions_data.energy_consumed

        output_tokens = generated_ids.shape[1] - input_tokens
        text = tokenizer.decode(generated_ids[0][input_tokens:], skip_special_tokens=True)
        out_f.write(f"--- Run {run} ---\n{text}\n\n")

        writer.writerow([run, input_tokens, output_tokens, f"{latency:.3f}",
                         f"{emissions_kg:.4e}", f"{energy_kwh:.4e}", ""])
        f.flush()

        print(f"Run {run}: {latency:.3f}s | {output_tokens} tokens | "
              f"{emissions_kg:.4e} kg CO2eq | {energy_kwh:.4e} kWh")

print(f"Docstrings saved to {outputs_file}")
print(f"Metrics saved to {results_csv}")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Version: quantized_4bit
Hardware: Tesla T4
Quantization: 4-bit NF4, double quant, FP16 compute
Model size on disk: 3.87 GB
GPU memory footprint: 3.76 GB


[codecarbon INFO @ 17:55:23] offline tracker init
[codecarbon WARNING @ 17:55:23] Multiple instances of codecarbon are allowed to run at the same time.


Run 1: 10.430s | 143 tokens | 8.2972e-05 kg CO2eq | 2.2457e-04 kWh
Run 2: 9.219s | 131 tokens | 7.4316e-05 kg CO2eq | 2.0114e-04 kWh
Run 3: 8.509s | 118 tokens | 6.8649e-05 kg CO2eq | 1.8580e-04 kWh
Run 4: 9.332s | 130 tokens | 7.5999e-05 kg CO2eq | 2.0570e-04 kWh
Run 5: 11.499s | 164 tokens | 9.3658e-05 kg CO2eq | 2.5349e-04 kWh
Docstrings saved to docstrings_quantized_4bit.txt
Metrics saved to results_quantized_4bit.csv


File downloads

In [4]:
from google.colab import files
files.download(results_csv)
files.download(outputs_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>